[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Expresiones en *Polars*.

## ¿Qué es una expresión?

Una **expresión** (`pl.Expr`) describe un cálculo que produce una columna a partir de otras. Por ejemplo, `pl.col('cantidad') * pl.col('precio_unitario')` describe "multiplicar la cantidad por el precio unitario", pero no lo calcula: la expresión se evalúa cuando se usa dentro de un contexto (`select()`, `with_columns()`, `filter()` o `group_by().agg()`), como se vio en el *notebook* 35.

Esta separación entre *describir* y *ejecutar* es la diferencia más importante con *Pandas*:

```python
# Pandas: cada línea se ejecuta de inmediato y modifica el dataframe.
df['importe'] = df['cantidad'] * df['precio_unitario']
df['importe_neto'] = df['importe'] * (1 - df['descuento'].fillna(0))

# Polars: se describen columnas nuevas y el contexto las evalúa.
df = df.with_columns(importe=pl.col('cantidad') * pl.col('precio_unitario'))
```

Como *Polars* conoce todas las expresiones de un contexto antes de ejecutarlas, puede evaluarlas en paralelo, evitar cálculos repetidos y, en el modo diferido (*notebook* 37), optimizar la consulta completa.

### Reglas de las expresiones.

* **Composición.** Una expresión se construye encadenando métodos: `pl.col('precio_unitario').mean().round(2)`.
* **Longitud.** Cada expresión de un contexto debe producir una columna del mismo largo que el *dataframe* o un solo valor. Un solo valor se repite en todos los renglones (*broadcasting*).
* **Nombre.** La columna resultante toma el nombre de la primera columna de la expresión. Se cambia con `alias()` o con los métodos de `name` (`name.suffix()`, `name.prefix()`).
* **Independencia.** Las expresiones de un mismo contexto no pueden usar columnas creadas en ese mismo contexto.

[Guía de expresiones](https://docs.pola.rs/user-guide/expressions/) · [Referencia](https://docs.pola.rs/api/python/stable/reference/expressions/index.html)

* La siguiente celda importa las bibliotecas y configura la presentación de flotantes.

In [ ]:
import os
from datetime import date

import polars as pl
import polars.selectors as cs

pl.Config.set_fmt_float('full')
print(f"Polars {pl.__version__}")

* La siguiente celda lee las ventas y calcula el importe neto de cada una, como en el *notebook* 35.

In [ ]:
ventas = (
    pl.read_csv(os.path.join('data', 'ventas.csv'), try_parse_dates=True)
    .with_columns(importe=pl.col('cantidad') * pl.col('precio_unitario'))
    .with_columns(importe_neto=(pl.col('importe') * (1 - pl.col('descuento').fill_null(0))).round(2))
)
ventas.select('fecha_hora', 'producto', 'cantidad', 'precio_unitario', 'descuento', 'importe_neto').head(3)

## Construcción de expresiones.

| Función | Descripción |
| :--- | :--- |
| [`pl.col(<nombres>)`](https://docs.pola.rs/api/python/stable/reference/expressions/col.html) | Una o varias columnas; también por tipo o por expresión regular (`^...$`). |
| [`pl.all()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.all.html) / [`pl.exclude(<nombres>)`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.exclude.html) | Todas las columnas / todas excepto las indicadas. |
| `cs.<selector>()` | Columnas por tipo o patrón de nombre (ver el *notebook* 35). |
| [`pl.lit(<valor>)`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.lit.html) | Un valor constante. |
| [`pl.len()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.len.html) | El número de renglones. |

Sobre una expresión se pueden aplicar:

* Operadores aritméticos (`+`, `-`, `*`, `/`, `//`, `%`, `**`) y de comparación (`==`, `!=`, `<`, `>`, ...).
* Métodos elemento a elemento, como `round()`, `abs()`, `cast()`, `fill_null()` o `clip()`.
* Agregaciones, que reducen la columna a un valor: `sum()`, `mean()`, `max()`, `n_unique()`, `quantile()`, ...
* Métodos de un espacio de nombres (*namespace*) según el tipo: `.str` para texto, `.dt` para fechas, `.list` para listas y `.struct` para registros.

**Ejemplos:**

* Una expresión con varias columnas se aplica a cada una. La siguiente celda obtiene el máximo de tres columnas y les agrega el sufijo `_max`.

In [ ]:
ventas.select(pl.col('cantidad', 'precio_unitario', 'importe_neto').max().name.suffix('_max'))

* La siguiente celda combina una columna con agregaciones. La suma total es un solo valor, que se repite en cada renglón para calcular la participación de cada venta en el total, en porcentaje.

In [ ]:
ventas.select(
    'venta_id',
    'importe_neto',
    (pl.col('importe_neto') / pl.col('importe_neto').sum() * 100).round(4).alias('participacion_pct'),
).head(3)

* La siguiente celda agrega columnas constantes con `pl.lit()` y cuenta los renglones con `pl.len()`.

In [ ]:
ventas.select(
    pl.lit('MXN').alias('moneda'),
    pl.len().alias('ventas'),
    pl.col('importe_neto').sum().round(2).alias('total'),
)

* La siguiente celda limita el descuento a un máximo de 10 % con `clip()` y aplica la misma operación de redondeo a varias columnas seleccionadas con un selector.

In [ ]:
ventas.select(
    pl.col('descuento').clip(upper_bound=0.10).alias('descuento_tope'),
    (cs.by_name('importe', 'importe_neto') / 1000).round(1).name.suffix('_miles'),
).head(3)

## Expresiones condicionales.

[`pl.when()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html) construye una expresión que toma distintos valores según una condición, de forma similar a `np.where()` en *NumPy* o a `CASE WHEN` en *SQL*.

```
pl.when(<condición_1>).then(<valor_1>)
  .when(<condición_2>).then(<valor_2>)
  .otherwise(<valor_por_omisión>)
```

Donde:

* Cada `<condición>` es una expresión booleana. Las condiciones se evalúan en orden y cada renglón toma el valor de la primera que se cumpla.
* Cada `<valor>` es una expresión o un valor constante. Las cadenas de texto se interpretan como **nombres de columnas**; para usar un texto constante se escribe `pl.lit('texto')`.
* `otherwise()` es opcional; si se omite, los renglones que no cumplen ninguna condición quedan como nulos.

**Ejemplos:**

* La siguiente celda clasifica cada venta según su importe neto.

In [ ]:
ventas.with_columns(
    pl.when(pl.col('importe_neto') >= 5000).then(pl.lit('alto'))
    .when(pl.col('importe_neto') >= 1000).then(pl.lit('medio'))
    .otherwise(pl.lit('bajo'))
    .alias('nivel')
).group_by('nivel').agg(pl.len()).sort('nivel')

* Los valores también pueden ser expresiones. La siguiente celda calcula una comisión del 3 % para las ventas en línea y del 5 % para las ventas en tienda.

In [ ]:
ventas.select(
    'canal',
    'importe_neto',
    pl.when(pl.col('canal') == 'en línea')
    .then(pl.col('importe_neto') * 0.03)
    .otherwise(pl.col('importe_neto') * 0.05)
    .round(2)
    .alias('comision'),
).head(4)

* Sin `otherwise()`, los renglones que no cumplen la condición quedan nulos.

In [ ]:
ventas.select(
    'descuento',
    pl.when(pl.col('descuento') > 0).then(pl.lit('con descuento')).alias('etiqueta'),
).head(4)

## Tipos de datos y conversiones.

*Polars* tiene un sistema de tipos basado en el de *Arrow*:

| Categoría | Tipos |
| :--- | :--- |
| Enteros | `Int8`, `Int16`, `Int32`, `Int64` y sus versiones sin signo `UInt8` ... `UInt64` |
| Punto flotante | `Float32`, `Float64` |
| Decimal | `Decimal(precision, scale)`, para cantidades de dinero sin errores de redondeo binario |
| Texto | `String` |
| Texto con valores limitados | `Categorical`, `Enum` |
| Booleano | `Boolean` |
| Tiempo | `Date`, `Datetime`, `Duration`, `Time` |
| Anidados | `List`, `Array`, `Struct` |
| Otros | `Binary`, `Null`, `Object` |

[Referencia de tipos](https://docs.pola.rs/api/python/stable/reference/datatypes.html)

### Conversión con `cast()`.

```
pl.col(<nombre>).cast(<tipo>, strict=<booleano>)
```

Donde:

* `<tipo>` es el tipo de destino.
* `strict=True` (por omisión) desencadena una excepción si algún valor no se puede convertir. Con `strict=False`, esos valores se convierten en nulos **sin aviso**.

*Polars* no convierte el texto en números ni los números en texto por su cuenta. Sí combina tipos numéricos en las operaciones aritméticas: un entero multiplicado por un flotante da un flotante.

**Ejemplos:**

* La siguiente celda convierte la cantidad a `Int8` y el precio a `Decimal` con dos decimales.

In [ ]:
ventas.select(
    pl.col('cantidad').cast(pl.Int8),
    pl.col('precio_unitario').cast(pl.Decimal(10, 2)),
).head(3)

* La siguiente celda intenta convertir a entero una columna con un texto no numérico. Se desencadenará una excepción de tipo `InvalidOperationError`.

In [ ]:
claves = pl.DataFrame({'clave': ['101', '102', 'S/N', '104']})
claves.select(pl.col('clave').cast(pl.Int64))

* Con `strict=False` la conversión no falla, pero el valor no convertible se vuelve nulo. Conviene revisar cuántos nulos se generan.

In [ ]:
convertidas = claves.with_columns(numero=pl.col('clave').cast(pl.Int64, strict=False))
print(f"Nulos generados: {convertidas['numero'].null_count()}")
convertidas

* La siguiente celda muestra la combinación de tipos numéricos: la cantidad (`Int64`) por el precio (`Float64`) da un `Float64`.

In [ ]:
ventas.select((pl.col('cantidad') * pl.col('precio_unitario')).alias('importe')).schema

## Los tipos `Categorical` y `Enum`.

Ambos tipos guardan cada valor distinto una sola vez y representan los renglones con enteros, lo que ahorra memoria cuando hay pocos valores distintos (países, categorías, estados de un proceso). Equivalen al tipo `category` de *Pandas*.

| Tipo | Valores permitidos | Orden |
| :--- | :--- | :--- |
| [`pl.Categorical`](https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Categorical.html) | Cualquier texto; las categorías se descubren al convertir. | Por valor de texto. |
| [`pl.Enum(<categorías>)`](https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Enum.html) | Sólo las categorías indicadas; cualquier otro valor es un error. | El de la lista de categorías. |

`Enum` es preferible cuando las categorías se conocen de antemano, porque valida los datos y permite ordenar con un orden propio (por ejemplo, *bajo*, *medio*, *alto*).

**Ejemplos:**

* La siguiente celda compara la memoria de la columna `categoria` como `String` y como `Categorical`.

In [ ]:
como_texto = ventas.select('categoria')
como_categoria = como_texto.with_columns(pl.col('categoria').cast(pl.Categorical))

print(f"String:      {como_texto.estimated_size('kb'):7.1f} KB")
print(f"Categorical: {como_categoria.estimated_size('kb'):7.1f} KB")

* La siguiente celda define un tipo `Enum` para el nivel de una venta y ordena por él: el orden es el de la lista de categorías, no el alfabético.

In [ ]:
nivel = pl.Enum(['bajo', 'medio', 'alto'])

pl.DataFrame({'nivel': ['alto', 'bajo', 'medio', 'bajo']}).with_columns(
    pl.col('nivel').cast(nivel)
).sort('nivel')

* La siguiente celda intenta convertir un valor que no está entre las categorías del `Enum`. Se desencadenará una excepción de tipo `InvalidOperationError`.

In [ ]:
pl.DataFrame({'nivel': ['alto', 'urgente']}).with_columns(pl.col('nivel').cast(nivel))

## El espacio de nombres `.str`.

Los métodos de texto se aplican con `pl.col(...).str.<método>()`. Los nombres difieren de los de *Pandas* en algunos casos:

| *Pandas* | *Polars* | Descripción |
| :--- | :--- | :--- |
| `.str.lower()` / `.str.upper()` | `.str.to_lowercase()` / `.str.to_uppercase()` | Minúsculas / mayúsculas. |
| `.str.strip()` | `.str.strip_chars()` | Elimina espacios al inicio y al final. |
| `.str.len()` | `.str.len_chars()` | Número de caracteres (`len_bytes()` cuenta bytes). |
| `.str.contains(<patrón>)` | `.str.contains(<patrón>)` | ¿Contiene el patrón? Es una expresión regular, salvo con `literal=True`. |
| `.str.startswith()` / `.str.endswith()` | `.str.starts_with()` / `.str.ends_with()` | ¿Empieza / termina con el texto? |
| `.str.replace(<patrón>, <reemplazo>)` | `.str.replace_all(<patrón>, <reemplazo>)` | Reemplaza **todas** las coincidencias. En *Polars*, `.str.replace()` reemplaza sólo la primera. |
| `.str.extract(<patrón>)` | `.str.extract(<patrón>, group_index=1)` | Extrae un grupo de captura de una expresión regular. |
| `.str.split(<separador>)` | `.str.split(<separador>)` | Divide en una lista (tipo `List(String)`). |
| `pd.to_datetime(<serie>, format=...)` | `.str.to_date(<formato>)` / `.str.to_datetime(<formato>)` | Convierte texto en fecha. |

[Referencia de `.str`](https://docs.pola.rs/api/python/stable/reference/expressions/string.html)

**Ejemplos:**

* La columna `cliente_email` tiene correos con mayúsculas y espacios de más. La siguiente celda muestra algunos de ellos, identificados porque contienen una mayúscula o empiezan o terminan con un espacio.

In [ ]:
ventas.filter(pl.col('cliente_email').str.contains(r'[A-Z]|^\s|\s$')).select('cliente_email').head(4)

* La siguiente celda limpia los correos y cuenta los clientes distintos antes y después de la limpieza.

In [ ]:
ventas = ventas.with_columns(
    cliente=pl.col('cliente_email').str.strip_chars().str.to_lowercase()
)
ventas.select(
    pl.col('cliente_email').n_unique().alias('antes'),
    pl.col('cliente').n_unique().alias('despues'),
)

* La siguiente celda extrae el dominio del correo con una expresión regular y el nombre de usuario con `split()` y `list.first()`.

In [ ]:
ventas.select(
    'cliente',
    pl.col('cliente').str.extract(r'@(.+)$', group_index=1).alias('dominio'),
    pl.col('cliente').str.split('@').list.first().alias('usuario'),
).head(3)

* La siguiente celda identifica los productos de electrónica cuyo nombre contiene `USB` y reemplaza los espacios por guiones bajos para generar una clave.

In [ ]:
ventas.select('producto').unique().sort('producto').with_columns(
    pl.col('producto').str.contains('USB', literal=True).alias('es_usb'),
    pl.col('producto').str.to_lowercase().str.replace_all(' ', '_').alias('clave'),
)

## El espacio de nombres `.dt`.

Los métodos de fecha y hora se aplican con `pl.col(...).dt.<método>()` sobre columnas `Date` o `Datetime`. A diferencia de *Pandas*, los componentes se obtienen con métodos (con paréntesis), no con atributos.

| *Pandas* | *Polars* | Descripción |
| :--- | :--- | :--- |
| `.dt.year`, `.dt.month`, `.dt.day`, `.dt.hour` | `.dt.year()`, `.dt.month()`, `.dt.day()`, `.dt.hour()` | Componentes de la fecha. |
| `.dt.dayofweek` (lunes = 0, domingo = 6) | `.dt.weekday()` (**lunes = 1, domingo = 7**) | Día de la semana. Las numeraciones difieren. |
| `.dt.date` | `.dt.date()` | Sólo la fecha, sin la hora. |
| `.dt.to_period('M')` | `.dt.truncate('1mo')` | Redondea hacia abajo a un periodo: `'1d'`, `'1w'`, `'1mo'`, `'1y'`, ... |
| `+ pd.DateOffset(months=1)` | `.dt.offset_by('1mo')` | Desplaza la fecha por un periodo de calendario. |
| `.dt.strftime(<formato>)` | `.dt.strftime(<formato>)` | Convierte en texto. |
| `(fin - inicio).dt.days` | `(fin - inicio).dt.total_days()` | Diferencia en días. |

[Referencia de `.dt`](https://docs.pola.rs/api/python/stable/reference/expressions/temporal.html)

**Ejemplos:**

* La siguiente celda extrae componentes de la fecha y hora de las primeras ventas. El 1 de enero de 2024 fue lunes, por lo que `weekday()` regresa `1`.

In [ ]:
ventas.select(
    'fecha_hora',
    pl.col('fecha_hora').dt.date().alias('fecha'),
    pl.col('fecha_hora').dt.hour().alias('hora'),
    pl.col('fecha_hora').dt.weekday().alias('dia_semana'),
    pl.col('fecha_hora').dt.truncate('1mo').dt.date().alias('mes'),
).head(3)

* La siguiente celda calcula el importe por día de la semana y muestra que las numeraciones de *Polars* y *Pandas* difieren en uno. El nombre del día se obtiene con `replace_strict()`, que sustituye cada valor según un diccionario.

In [ ]:
dias = {1: 'lunes', 2: 'martes', 3: 'miércoles', 4: 'jueves', 5: 'viernes', 6: 'sábado', 7: 'domingo'}

(
    ventas
    .group_by(pl.col('fecha_hora').dt.weekday().alias('weekday'))
    .agg(pl.col('importe_neto').sum().round(2))
    .sort('weekday')
    .with_columns(
        dia=pl.col('weekday').replace_strict(dias, return_dtype=pl.String),
        dayofweek_pandas=pl.col('weekday') - 1,
    )
)

* La siguiente celda calcula, para las tiendas del catálogo, los días transcurridos desde su apertura hasta el 31 de diciembre de 2025 y la fecha de su primer aniversario.

In [ ]:
tiendas = pl.read_csv(os.path.join('data', 'tiendas.csv'), try_parse_dates=True)
corte = date(2025, 12, 31)

tiendas.select(
    'nombre',
    'fecha_apertura',
    (pl.lit(corte) - pl.col('fecha_apertura')).dt.total_days().alias('dias_abierta'),
    pl.col('fecha_apertura').dt.offset_by('1y').alias('primer_aniversario'),
).head(4)

* La siguiente celda suma el importe por mes con `group_by_dynamic()`, que agrupa por intervalos de tiempo. Requiere que los datos estén ordenados por la columna de tiempo.

In [ ]:
(
    ventas
    .sort('fecha_hora')
    .group_by_dynamic('fecha_hora', every='1mo')
    .agg(pl.col('importe_neto').sum().round(2).alias('importe_mes'))
    .head(6)
)

## Listas y *structs*.

Algunas operaciones producen columnas anidadas (ver los tipos anidados de *Arrow* en el *notebook* 34):

* Una columna `List` contiene una lista por renglón. Sus métodos están en [`.list`](https://docs.pola.rs/api/python/stable/reference/expressions/list.html): `len()`, `first()`, `get(<índice>)`, `contains()`, `join(<separador>)`, ...
* Una columna `Struct` contiene un registro con campos por renglón. Sus métodos están en [`.struct`](https://docs.pola.rs/api/python/stable/reference/expressions/struct.html): `field(<nombre>)`, `unnest()`, ...
* `df.explode(<columna>)` convierte cada elemento de una lista en un renglón, y `df.unnest(<columna>)` convierte cada campo de un *struct* en una columna.

**Ejemplos:**

* La siguiente celda divide el nombre de cada producto en palabras y obtiene el número de palabras y la última.

In [ ]:
productos = ventas.select('producto').unique().sort('producto').with_columns(
    palabras=pl.col('producto').str.split(' ')
)
productos.with_columns(
    n_palabras=pl.col('palabras').list.len(),
    ultima=pl.col('palabras').list.last(),
).head(4)

* La siguiente celda convierte cada palabra en un renglón con `explode()` y cuenta las palabras más frecuentes.

In [ ]:
productos.explode('palabras').group_by('palabras').agg(pl.len()).sort('len', descending=True).head(3)

* `value_counts()` regresa un *struct* con el valor y su frecuencia. La siguiente celda lo separa en columnas con `unnest()`.

In [ ]:
conteo = ventas.select(pl.col('canal').value_counts(sort=True))
print(conteo.schema)
conteo.unnest('canal')

## Funciones de ventana.

Una agregación dentro de `group_by().agg()` produce un renglón por grupo. Con [`over()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.over.html), la misma agregación se calcula por grupo pero el resultado **conserva todos los renglones**: cada renglón recibe el valor de su grupo. Equivale a `transform()` en *Pandas* y a `OVER (PARTITION BY ...)` en *SQL*.

```
<expresión>.over(<columnas>, order_by=<columna>)
```

Donde:

* `<columnas>` define los grupos (*particiones*).
* `order_by` es opcional y define el orden de los renglones dentro de cada grupo. Es necesario para las operaciones que dependen del orden, como `cum_sum()`, `shift()` o `rank()` con desempates; si se omite, se usa el orden actual del *dataframe*.

| Expresión | Resultado por renglón |
| :--- | :--- |
| `pl.col(...).sum().over(...)` | Total del grupo. |
| `pl.col(...).cum_sum().over(..., order_by=...)` | Suma acumulada dentro del grupo. |
| `pl.col(...).rank(method=...).over(...)` | Posición dentro del grupo; `method` puede ser `'dense'`, `'min'`, `'ordinal'`, ... |
| `pl.col(...).shift(n).over(..., order_by=...)` | Valor `n` renglones antes (`n` positivo) o después (`n` negativo) dentro del grupo. |

**Ejemplos:**

* La siguiente celda calcula la participación de cada venta en el importe total de su tienda.

In [ ]:
ventas.select(
    'venta_id',
    'tienda_id',
    'importe_neto',
    (pl.col('importe_neto') / pl.col('importe_neto').sum().over('tienda_id') * 100)
    .round(3)
    .alias('pct_de_su_tienda'),
).head(4)

* La siguiente celda calcula el importe acumulado de cada tienda en orden cronológico, indicado con `order_by`, y lo muestra para la tienda 1.

In [ ]:
(
    ventas
    .with_columns(
        acumulado_tienda=pl.col('importe_neto').cum_sum().over('tienda_id', order_by='fecha_hora').round(2)
    )
    .filter(pl.col('tienda_id') == 1)
    .select('fecha_hora', 'tienda_id', 'importe_neto', 'acumulado_tienda')
    .head(4)
)

* La siguiente celda calcula el importe mensual de cada categoría y la variación respecto del mes anterior con `shift(1)` dentro de cada categoría.

In [ ]:
mensual = (
    ventas
    .group_by('categoria', pl.col('fecha_hora').dt.truncate('1mo').dt.date().alias('mes'))
    .agg(pl.col('importe_neto').sum().round(2).alias('importe'))
)

mensual.with_columns(
    mes_anterior=pl.col('importe').shift(1).over('categoria', order_by='mes'),
).with_columns(
    variacion_pct=((pl.col('importe') / pl.col('mes_anterior') - 1) * 100).round(1),
).filter(pl.col('categoria') == 'Hogar').sort('mes').head(4)

* La siguiente celda obtiene, con `rank()`, los dos productos más vendidos (en piezas) de cada categoría.

In [ ]:
(
    ventas
    .group_by('categoria', 'producto')
    .agg(pl.col('cantidad').sum().alias('piezas'))
    .with_columns(lugar=pl.col('piezas').rank(method='ordinal', descending=True).over('categoria'))
    .filter(pl.col('lugar') <= 2)
    .sort('categoria', 'lugar')
)

## Guía de traducción de *Pandas* 3 a *Polars*.

| Operación | *Pandas* 3 | *Polars* |
| :--- | :--- | :--- |
| Seleccionar columnas | `df[['a', 'b']]` | `df.select('a', 'b')` |
| Filtrar renglones | `df[df['a'] > 5]` o `df.query('a > 5')` | `df.filter(pl.col('a') > 5)` |
| Agregar una columna | `df.assign(c=df['a'] * 2)` o `df.assign(c=pd.col('a') * 2)` | `df.with_columns(c=pl.col('a') * 2)` |
| Condicional | `np.where(cond, x, y)` | `pl.when(cond).then(x).otherwise(y)` |
| Rellenar nulos | `df['a'].fillna(0)` | `pl.col('a').fill_null(0)` |
| Agrupar y agregar | `df.groupby('g').agg(total=('a', 'sum'))` | `df.group_by('g').agg(total=pl.col('a').sum())` |
| Total del grupo en cada renglón | `df.groupby('g')['a'].transform('sum')` | `pl.col('a').sum().over('g')` |
| Ordenar | `df.sort_values('a', ascending=False)` | `df.sort('a', descending=True)` |
| Combinar tablas | `df.merge(otro, on='k', how='left')` | `df.join(otro, on='k', how='left')` |
| Tabla dinámica | `df.pivot_table(index=..., columns=..., values=...)` | `df.pivot(on=..., index=..., values=...)` |
| Texto en minúsculas | `.str.lower()` | `.str.to_lowercase()` |
| Día de la semana | `.dt.dayofweek` (lunes = 0) | `.dt.weekday()` (lunes = 1) |
| Tipo categórico | `.astype('category')` | `.cast(pl.Categorical)` o `.cast(pl.Enum([...]))` |

[Guía de migración desde *Pandas*](https://docs.pola.rs/user-guide/migration/pandas/)

**Ejemplo:**

* La siguiente celda calcula, con *Pandas* y con *Polars*, el importe neto total de las ventas en línea por región, a partir de las ventas y del catálogo de tiendas. Ambos resultados coinciden.

In [ ]:
import pandas as pd

# Pandas
ventas_pd = ventas.to_pandas()
tiendas_pd = tiendas.to_pandas()
resultado_pd = (
    ventas_pd[ventas_pd['canal'] == 'en línea']
    .merge(tiendas_pd[['tienda_id', 'region']], on='tienda_id', how='inner')
    .groupby('region', as_index=False)
    .agg(importe=('importe_neto', 'sum'))
    .sort_values('importe', ascending=False)
)

# Polars
resultado_pl = (
    ventas
    .filter(pl.col('canal') == 'en línea')
    .join(tiendas.select('tienda_id', 'region'), on='tienda_id', how='inner')
    .group_by('region')
    .agg(importe=pl.col('importe_neto').sum())
    .sort('importe', descending=True)
)

print(resultado_pd.round(2).to_string(index=False))
print()
print(resultado_pl.with_columns(pl.col('importe').round(2)))

## Resumen.

| Concepto | Descripción |
| :--- | :--- |
| **Expresión** | Descripción de un cálculo sobre columnas; se evalúa dentro de un contexto. |
| **Reglas** | Las expresiones se componen con métodos, producen una columna del largo del *dataframe* o un valor, y toman el nombre de la primera columna. |
| **`pl.when().then().otherwise()`** | Valores condicionales; el texto constante va en `pl.lit()`. |
| **Tipos y `cast()`** | Tipos estrictos; `strict=False` convierte en nulo lo que no se puede convertir. |
| **`Categorical` y `Enum`** | Texto con pocos valores distintos; `Enum` valida y define el orden. |
| **`.str`, `.dt`, `.list`, `.struct`** | Métodos de texto, tiempo, listas y registros. En `.dt.weekday()`, lunes = 1. |
| **Funciones de ventana** | `over()` calcula por grupo sin reducir renglones; `order_by` fija el orden dentro del grupo. |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>